# 🔄 Lab 02 — Transformations SQL & Requêtes Planifiées sur BigQuery

**Objectifs pédagogiques :**
- Implémenter l'architecture Medallion (Bronze → Silver → Gold) avec du SQL natif BigQuery
- Utiliser les **Vues** et **Tables Matérialisées** comme couches de transformation
- Automatiser les transformations avec les **Scheduled Queries** (requêtes planifiées)
- Comparer les coûts : scan full table vs table partitionnée + clustered

**Durée estimée :** 45 min  
**Outils :** Console BigQuery, `bq` CLI (Cloud Shell ou terminal local)

---

> 💡 **Convention :**  
> - 🟨 Cellules `sql` → à exécuter dans la **Console BigQuery** (copier/coller)  
> - 🟦 Cellules `bash` → à exécuter dans le **terminal** (Cloud Shell ou local)


---
## ⚙️ Partie 0 — Préparation de l'environnement

### 0.1 — Créer les datasets (couches Medallion)


In [ ]:
export PROJECT_ID=$(gcloud config get-value project)
echo "PROJECT_ID = $PROJECT_ID"

# Créer les 3 couches de l'architecture Medallion
for LAYER in bronze silver gold; do
  bq mk --location=EU --dataset "${PROJECT_ID}:${LAYER}"
  echo "✅ Dataset créé : ${LAYER}"
done


### 0.2 — Créer la table source brute (couche Bronze Landing)

Cette table simule une ingestion de données d'événements e-commerce depuis une API externe.


In [ ]:
-- ── Console BigQuery ──────────────────────────────────────────────────────
-- Créer la table source brute (données non nettoyées, as-is)
CREATE OR REPLACE TABLE `bronze.raw_events`
(
  event_id     STRING,
  event_ts     STRING,        -- timestamp brut sous forme de STRING
  user_id      STRING,
  event_type   STRING,
  device_os    STRING,
  amount_str   STRING,        -- montant brut sous forme de STRING
  country      STRING,
  _loaded_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP()  -- métadonnée d'ingestion
)
PARTITION BY DATE(_loaded_at)  -- partitionnement par date d'ingestion
OPTIONS (
  description = 'Événements bruts ingérés — couche Bronze, jamais modifiée'
);


In [ ]:
-- Insérer des données de test (incluant des doublons et valeurs malformées)
INSERT INTO `bronze.raw_events`
  (event_id, event_ts, user_id, event_type, device_os, amount_str, country)
VALUES
  ('evt-001', '2024-01-15T10:23:45Z', 'usr-42', 'page_view',        'Android', NULL,    'FR'),
  ('evt-002', '2024-01-15T10:25:01Z', 'usr-42', 'purchase_success', 'Android', '49.99', 'FR'),
  ('evt-003', '2024-01-15T11:00:00Z', 'usr-17', 'page_view',        'iOS',     NULL,    'BE'),
  ('evt-004', '2024-01-15T11:05:22Z', 'usr-17', 'purchase_success', 'iOS',     '129.0', 'BE'),
  ('evt-005', '2024-01-16T09:10:00Z', 'usr-99', 'purchase_success', 'Windows', '25.50', 'FR'),
  ('evt-006', '2024-01-16T14:30:00Z', 'usr-42', 'page_view',        'Android', NULL,    'FR'),
  ('evt-007', '2024-01-16T14:45:10Z', 'usr-42', 'add_to_cart',      'Android', '89.00', 'FR'),
  -- Doublon intentionnel (même event_id, timestamp différent)
  ('evt-002', '2024-01-15T10:26:00Z', 'usr-42', 'purchase_success', 'Android', '49.99', 'FR'),
  -- Valeur invalide
  ('evt-008', '2024-01-16T16:00:00Z', 'usr-55', 'purchase_success', 'UnknownOS', 'N/A', 'XX');


In [ ]:
-- Vérifier les données brutes (9 lignes dont 1 doublon et 1 ligne invalide)
SELECT
  event_id,
  event_ts,
  user_id,
  event_type,
  device_os,
  amount_str,
  country
FROM `bronze.raw_events`
ORDER BY event_ts;


---
## 🥈 Partie 1 — Couche Silver : Nettoyage par requête SQL

La couche Silver **nettoie, type, filtre et déduplique** les données brutes.  
Nous la créons d'abord manuellement pour comprendre la logique, puis nous l'automatiserons.

### 1.1 — Explorer les problèmes de qualité dans le Bronze


In [ ]:
-- Diagnostic : compter les doublons
SELECT
  event_id,
  COUNT(*) AS occurrences
FROM `bronze.raw_events`
GROUP BY 1
HAVING COUNT(*) > 1;
-- → evt-002 apparaît 2 fois


In [ ]:
-- Diagnostic : valeurs invalides dans device_os et amount_str
SELECT
  device_os,
  amount_str,
  COUNT(*) AS nb
FROM `bronze.raw_events`
GROUP BY 1, 2
ORDER BY 1;


### 1.2 — Créer la table Silver avec une requête de transformation


In [ ]:
-- Transformation Bronze → Silver
-- Logique : typage, déduplication, filtrage des valeurs invalides
CREATE OR REPLACE TABLE `silver.events`
PARTITION BY event_date
CLUSTER BY user_id, event_type
OPTIONS (
  description = 'Événements nettoyés et dédupliqués — couche Silver'
)
AS
WITH
-- Étape 1 : typer et normaliser les colonnes
typed AS (
  SELECT
    event_id,
    PARSE_TIMESTAMP('%Y-%m-%dT%H:%M:%SZ', event_ts)   AS event_ts,
    user_id,
    LOWER(TRIM(event_type))                            AS event_type,
    device_os,
    SAFE_CAST(amount_str AS FLOAT64)                   AS amount,  -- NULL si invalide
    country,
    _loaded_at
  FROM `bronze.raw_events`
  WHERE
    -- Filtrer les lignes clairement invalides
    device_os NOT IN ('UnknownOS')
    AND country != 'XX'
),
-- Étape 2 : dédupliquer (garder l'occurrence la plus récente par event_id)
deduped AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY event_id
      ORDER BY event_ts DESC
    ) AS rn
  FROM typed
)
SELECT
  event_id,
  event_ts,
  DATE(event_ts)  AS event_date,
  user_id,
  event_type,
  device_os,
  amount,
  country,
  _loaded_at
FROM deduped
WHERE rn = 1;  -- garder seulement l'enregistrement le plus récent


In [ ]:
-- Vérifier le résultat Silver
-- Attendu : 7 lignes (9 - 1 doublon - 1 invalide)
SELECT
  event_id,
  event_date,
  user_id,
  event_type,
  device_os,
  amount,
  country
FROM `silver.events`
ORDER BY event_ts;


In [ ]:
-- Comparer Bronze vs Silver
SELECT 'Bronze (raw)' AS couche, COUNT(*) AS lignes FROM `bronze.raw_events`
UNION ALL
SELECT 'Silver (clean)', COUNT(*) FROM `silver.events`;


---
## 🔄 Partie 2 — Tables Matérialisées BigQuery

Une **Materialized View** (vue matérialisée) est une requête pré-calculée et stockée par BigQuery.  
Elle se met à jour **automatiquement** quand la table source change — sans orchestrateur externe !

### 2.1 — Créer une vue matérialisée Silver (alternative à la table)


In [ ]:
-- Option alternative : Vue Matérialisée au lieu d'une table
-- BigQuery la rafraîchit automatiquement lors de chaque requête (max toutes les 5 min)
CREATE MATERIALIZED VIEW IF NOT EXISTS `silver.mv_events`
PARTITION BY event_date
CLUSTER BY user_id
OPTIONS (
  enable_refresh = true,
  refresh_interval_minutes = 30,
  description = 'Vue matérialisée des événements nettoyés (auto-refresh 30 min)'
)
AS
SELECT
  event_id,
  PARSE_TIMESTAMP('%Y-%m-%dT%H:%M:%SZ', event_ts)   AS event_ts,
  DATE(PARSE_TIMESTAMP('%Y-%m-%dT%H:%M:%SZ', event_ts)) AS event_date,
  user_id,
  LOWER(TRIM(event_type))                            AS event_type,
  device_os,
  SAFE_CAST(amount_str AS FLOAT64)                   AS amount,
  country
FROM `bronze.raw_events`
WHERE
  device_os != 'UnknownOS'
  AND country != 'XX';
-- Note : les fenêtres ROW_NUMBER() ne sont pas supportées dans les MV
-- → Pour la déduplication complexe, préférer une Scheduled Query


### 2.2 — Quand utiliser quoi ?

| Type | Rafraîchissement | Déduplication | Coût stockage | Idéal pour |
|------|-----------------|---------------|--------------|------------|
| **Table** | Manuel / Scheduled | ✅ Oui (ROW_NUMBER) | Stockage plein | Silver/Gold avec logique complexe |
| **Vue** | À chaque requête | ✅ Oui | Aucun | Bronze léger, exploration |
| **Materialized View** | Automatique (~30 min) | ❌ Non (pas de fenêtres) | Partiel | Agrégations Gold simples |


---
## 🥇 Partie 3 — Couche Gold : Agrégations BI

La couche Gold contient les **métriques pré-calculées** consommées par Looker Studio ou d'autres outils.

### 3.1 — Créer la table Gold


In [ ]:
-- Transformation Silver → Gold
-- KPIs quotidiens par pays et OS — prêts pour Looker Studio
CREATE OR REPLACE TABLE `gold.daily_kpis`
PARTITION BY report_date
CLUSTER BY country
OPTIONS (
  description = 'KPIs quotidiens agrégés — couche Gold, source Looker Studio'
)
AS
SELECT
  event_date                                                  AS report_date,
  country,
  device_os,
  COUNT(DISTINCT user_id)                                     AS active_users,
  COUNT(*)                                                    AS total_events,
  COUNT(CASE WHEN event_type = 'page_view'        THEN 1 END) AS page_views,
  COUNT(CASE WHEN event_type = 'purchase_success' THEN 1 END) AS purchases,
  COUNT(CASE WHEN event_type = 'add_to_cart'      THEN 1 END) AS cart_adds,
  ROUND(SUM(amount), 2)                                       AS total_revenue,
  ROUND(AVG(CASE WHEN event_type = 'purchase_success'
                 THEN amount END), 2)                        AS avg_basket
FROM `silver.events`
GROUP BY 1, 2, 3
ORDER BY 1 DESC, 2, 3;


In [ ]:
-- Vérifier le résultat Gold
SELECT * FROM `gold.daily_kpis` ORDER BY report_date DESC;


---
## ⏰ Partie 4 — Requêtes Planifiées (Scheduled Queries)

Les **Scheduled Queries** de BigQuery permettent d'exécuter automatiquement une requête SQL  
sur un calendrier défini (toutes les heures, chaque nuit, etc.) — **sans serveur, sans orchestrateur**.

### 4.1 — Créer une Scheduled Query via la Console BigQuery

1. Ouvrez la **Console BigQuery** → [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery)
2. Collez la requête ci-dessous dans l'éditeur
3. Cliquez sur **"Planifier"** → **"Créer une requête planifiée"**
4. Configurez :
   - **Nom** : `refresh_silver_events`
   - **Récurrence** : `Toutes les heures` (ou `Chaque jour à 02:00`)
   - **Dataset de destination** : `silver`
   - **Table de destination** : `events`
   - **Mode d'écriture** : `Overwrite table` (ou `Append to table` si incrémental)


In [ ]:
-- ── Requête à planifier : Silver refresh ────────────────────────────────────
-- Coller dans l'éditeur BigQuery avant de cliquer sur "Planifier"
CREATE OR REPLACE TABLE `silver.events`
PARTITION BY event_date
CLUSTER BY user_id, event_type
AS
WITH typed AS (
  SELECT
    event_id,
    PARSE_TIMESTAMP('%Y-%m-%dT%H:%M:%SZ', event_ts)   AS event_ts,
    user_id,
    LOWER(TRIM(event_type))                            AS event_type,
    device_os,
    SAFE_CAST(amount_str AS FLOAT64)                   AS amount,
    country,
    _loaded_at
  FROM `bronze.raw_events`
  WHERE device_os NOT IN ('UnknownOS') AND country != 'XX'
),
deduped AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY event_ts DESC) AS rn
  FROM typed
)
SELECT
  event_id, event_ts, DATE(event_ts) AS event_date,
  user_id, event_type, device_os, amount, country, _loaded_at
FROM deduped
WHERE rn = 1;


### 4.2 — Créer une Scheduled Query via le CLI (`bq`)

Alternative à la Console : créer la requête planifiée en ligne de commande.


In [ ]:
export PROJECT_ID=$(gcloud config get-value project)

# Créer une Scheduled Query qui rafraîchit Silver toutes les heures
bq mk \
  --transfer_config \
  --project_id="${PROJECT_ID}" \
  --data_source=scheduled_query \
  --target_dataset=silver \
  --display_name="Refresh Silver Events (hourly)" \
  --schedule="every 60 minutes" \
  --params='{
    "query": "CREATE OR REPLACE TABLE silver.events PARTITION BY event_date CLUSTER BY user_id AS SELECT event_id, PARSE_TIMESTAMP(\"%Y-%m-%dT%H:%M:%SZ\", event_ts) AS event_ts, DATE(PARSE_TIMESTAMP(\"%Y-%m-%dT%H:%M:%SZ\", event_ts)) AS event_date, user_id, LOWER(TRIM(event_type)) AS event_type, device_os, SAFE_CAST(amount_str AS FLOAT64) AS amount, country FROM bronze.raw_events WHERE device_os NOT IN (\"UnknownOS\") AND country != \"XX\""
  }'


In [ ]:
# Lister toutes les Scheduled Queries du projet
bq ls --transfer_config --transfer_location=EU --project_id="${PROJECT_ID}"


### 4.3 — Déclencher manuellement une Scheduled Query

Utile pour tester sans attendre l'heure planifiée.


In [ ]:
# Récupérer l'ID de la config créée
TRANSFER_NAME=$(bq ls --transfer_config --transfer_location=EU   --project_id="${PROJECT_ID}" --format=json |   python3 -c "import sys,json; configs=json.load(sys.stdin);   [print(c['name']) for c in configs if 'Refresh Silver' in c.get('displayName','')]")

echo "Transfer config : $TRANSFER_NAME"

# Déclencher manuellement une exécution
bq mk --transfer_run \
  --start_time=$(date -u +%Y-%m-%dT%H:%M:%SZ) \
  --end_time=$(date -u +%Y-%m-%dT%H:%M:%SZ) \
  $TRANSFER_NAME


---
## 📊 Partie 5 — Benchmark : Partitionnement & Clustering

Maintenant que nos tables Silver et Gold sont **partitionnées et clusterisées**,  
comparons concrètement l'impact sur les bytes scannés (= coût BigQuery).

### 5.1 — Simuler un grand volume de données


In [ ]:
-- Générer ~100k lignes en répétant les données de test
-- (utilise CROSS JOIN pour multiplier les lignes)
INSERT INTO `bronze.raw_events` (event_id, event_ts, user_id, event_type, device_os, amount_str, country)
SELECT
  CONCAT('evt-gen-', CAST(ROW_NUMBER() OVER () AS STRING)) AS event_id,
  FORMAT_TIMESTAMP('%Y-%m-%dT%H:%M:%SZ',
    TIMESTAMP_ADD('2024-01-01T00:00:00Z',
      INTERVAL MOD(ROW_NUMBER() OVER (), 86400) * 60 SECOND)) AS event_ts,
  CONCAT('usr-', CAST(MOD(ROW_NUMBER() OVER (), 500) AS STRING)) AS user_id,
  event_type,
  device_os,
  amount_str,
  country
FROM `bronze.raw_events`
CROSS JOIN UNNEST(GENERATE_ARRAY(1, 2000)) AS n;  -- ~18 000 lignes

-- Re-créer Silver avec les nouvelles données
-- (ré-exécuter la requête de la Partie 1.2)


### 5.2 — Comparer scan sans filtre vs avec filtre sur partition


In [ ]:
-- ❌ Requête NON optimisée : scan de TOUTE la table
-- → Avant d'exécuter, notez les "Bytes à traiter" affichés par BigQuery
SELECT
  COUNT(*) AS total,
  ROUND(SUM(amount), 2) AS revenue
FROM `silver.events`;


In [ ]:
-- ✅ Requête OPTIMISÉE : filtre sur la colonne de partition (event_date)
-- → Comparer les "Bytes à traiter" avec la requête précédente
SELECT
  COUNT(*) AS total,
  ROUND(SUM(amount), 2) AS revenue
FROM `silver.events`
WHERE event_date BETWEEN '2024-01-15' AND '2024-01-16';


In [ ]:
-- ✅ Requête OPTIMISÉE avec partition + cluster (user_id est clustered)
-- → BigQuery lit seulement les blocs correspondant au user_id
SELECT
  event_date,
  event_type,
  amount
FROM `silver.events`
WHERE event_date = '2024-01-15'
  AND user_id = 'usr-42'   -- cluster column → scan minimal
ORDER BY event_ts;


### 5.3 — Tableau de résultats à compléter

Relevez les **"Bytes traités"** affichés dans la Console BigQuery avant chaque exécution :

| Requête | Filtre | Bytes traités | Coût estimé ($/TB = $5) |
|---------|--------|--------------|------------------------|
| Full scan | Aucun | ___ MB | ___ |
| Filtre partition | `event_date BETWEEN ...` | ___ MB | ___ |
| Partition + Cluster | `event_date + user_id` | ___ MB | ___ |

> 💡 **Règle :** 1 TB scanné = ~5$ sur BigQuery on-demand. Le partitionnement peut réduire les coûts de 90%+.


---
## ✅ Récapitulatif

### Ce que vous avez construit

```
bronze.raw_events          (table partitionnée, données brutes as-is)
         │
         │  SQL de transformation  ←── Scheduled Query (toutes les heures)
         ▼
silver.events              (table partitionnée + clustered, nettoyée + dédupliquée)
         │
         │  SQL d'agrégation       ←── Scheduled Query (chaque nuit)
         ▼
gold.daily_kpis            (table partitionnée + clustered, KPIs prêts pour BI)
         │
         ▼
   Looker Studio
```

### Outils utilisés

| Outil BigQuery | Rôle |
|----------------|------|
| `CREATE TABLE ... AS SELECT` | Matérialiser une transformation |
| `PARTITION BY` | Réduire les bytes scannés (= coût) |
| `CLUSTER BY` | Optimiser les filtres fréquents |
| `Scheduled Queries` | Automatiser sans orchestrateur |
| `MATERIALIZED VIEW` | Auto-refresh pour agrégations simples |
| `SAFE_CAST` | Gérer les valeurs malformées sans erreur |
| `ROW_NUMBER() OVER(PARTITION BY ...)` | Déduplication propre |

### 🧠 Questions de réflexion

1. Quand utiliserait-on une **Scheduled Query** plutôt qu'une **Materialized View** ?
2. Pourquoi partitionner par `event_date` plutôt que par `user_id` ?
3. Si de nouvelles données arrivent en continu (streaming), quelle approche choisir ?
4. Comment gérer les **données en retard** (*late data*) qui arrivent le lendemain ?
